In [1]:
import pandas as pd

# Load all 6 tables
orders   = pd.read_csv('/content/olist_orders_dataset.csv')
items    = pd.read_csv('/content/olist_order_items_dataset.csv')
sellers  = pd.read_csv('/content/olist_sellers_dataset.csv')
products = pd.read_csv('/content/olist_products_dataset.csv')
reviews  = pd.read_csv('/content/olist_order_reviews_dataset.csv')
customers = pd.read_csv('/content/olist_customers_dataset.csv')

# Check shapes
tables = {
    'orders': orders,
    'items': items,
    'sellers': sellers,
    'products': products,
    'reviews': reviews,
    'customers': customers
}

for name, df in tables.items():
    print(f"{name}: {df.shape} — {df.columns.tolist()}")

orders: (99441, 8) — ['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']
items: (112650, 7) — ['order_id', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value']
sellers: (3095, 4) — ['seller_id', 'seller_zip_code_prefix', 'seller_city', 'seller_state']
products: (32951, 9) — ['product_id', 'product_category_name', 'product_name_lenght', 'product_description_lenght', 'product_photos_qty', 'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm']
reviews: (99224, 7) — ['review_id', 'order_id', 'review_score', 'review_comment_title', 'review_comment_message', 'review_creation_date', 'review_answer_timestamp']
customers: (99441, 5) — ['customer_id', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state']


In [2]:
# --- 1. Check orders table in detail ---
print("=== ORDERS TABLE ===")
print(orders.dtypes)
print("\nNull counts:")
print(orders.isnull().sum())
print("\nOrder status distribution:")
print(orders['order_status'].value_counts())
print("\nSample timestamps:")
print(orders[['order_id','order_status',
              'order_purchase_timestamp',
              'order_approved_at',
              'order_delivered_carrier_date',
              'order_delivered_customer_date',
              'order_estimated_delivery_date']].head())

# --- 2. Check items table ---
print("\n=== ITEMS TABLE ===")
print(items.dtypes)
print("\nNull counts:")
print(items.isnull().sum())
print("\nSample:")
print(items[['order_id','seller_id','price','freight_value']].head())

# --- 3. Check reviews ---
print("\n=== REVIEWS TABLE ===")
print(reviews['review_score'].value_counts().sort_index())
print("\nNull counts:")
print(reviews[['order_id','review_score']].isnull().sum())

# --- 4. Check products ---
print("\n=== PRODUCTS TABLE ===")
print(products['product_category_name'].value_counts().head(10))
print("\nNull counts in category:")
print(products['product_category_name'].isnull().sum())

=== ORDERS TABLE ===
order_id                         object
customer_id                      object
order_status                     object
order_purchase_timestamp         object
order_approved_at                object
order_delivered_carrier_date     object
order_delivered_customer_date    object
order_estimated_delivery_date    object
dtype: object

Null counts:
order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64

Order status distribution:
order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

Sample timestamps:
                           order_id order_status order_purc

In [3]:
print("Orders shape:", orders.shape)
print("\nOrders dtypes:")
print(orders.dtypes)
print("\nOrders NULL counts:")
print(orders.isnull().sum())
print("\nOrder status counts:")
print(orders['order_status'].value_counts())
print("\nSample delivery timestamps:")
print(orders[['order_purchase_timestamp',
              'order_delivered_customer_date',
              'order_estimated_delivery_date']].head(3))


Orders shape: (99441, 8)

Orders dtypes:
order_id                         object
customer_id                      object
order_status                     object
order_purchase_timestamp         object
order_approved_at                object
order_delivered_carrier_date     object
order_delivered_customer_date    object
order_estimated_delivery_date    object
dtype: object

Orders NULL counts:
order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64

Order status counts:
order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

Sample delivery timestamps:
  order_purchase_timestamp or

In [4]:
import pandas as pd
import numpy as np

# --- 1. Convert all timestamp columns to datetime ---
timestamp_cols = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]

for col in timestamp_cols:
    orders[col] = pd.to_datetime(orders[col])

print("Timestamps converted successfully")

# --- 2. Filter to delivered orders only ---
# Only delivered orders have meaningful SLA data
orders_delivered = orders[orders['order_status'] == 'delivered'].copy()
print(f"Delivered orders: {len(orders_delivered):,}")

# --- 3. Drop rows missing critical delivery timestamps ---
orders_clean = orders_delivered.dropna(subset=[
    'order_delivered_customer_date',
    'order_estimated_delivery_date',
    'order_delivered_carrier_date'
])
print(f"After dropping missing delivery dates: {len(orders_clean):,}")

# --- 4. Engineer delivery metrics ---
# Days from purchase to actual delivery
orders_clean['days_to_deliver'] = (
    orders_clean['order_delivered_customer_date'] -
    orders_clean['order_purchase_timestamp']
).dt.days

# SLA delay: positive = late, negative = early
orders_clean['delay_days'] = (
    orders_clean['order_delivered_customer_date'] -
    orders_clean['order_estimated_delivery_date']
).dt.days

# Time from purchase to carrier pickup (seller processing time)
orders_clean['seller_processing_days'] = (
    orders_clean['order_delivered_carrier_date'] -
    orders_clean['order_purchase_timestamp']
).dt.days

# Time from carrier pickup to customer (transit time)
orders_clean['transit_days'] = (
    orders_clean['order_delivered_customer_date'] -
    orders_clean['order_delivered_carrier_date']
).dt.days

# Time from purchase to approval (payment processing)
orders_clean['approval_days'] = (
    orders_clean['order_approved_at'] -
    orders_clean['order_purchase_timestamp']
).dt.days

# --- 5. SLA classification ---
orders_clean['sla_status'] = pd.cut(
    orders_clean['delay_days'],
    bins=[-999, -1, 2, 999],
    labels=['On Time', 'At Risk', 'Breached']
)

# --- 6. Extract time dimensions ---
orders_clean['purchase_year']  = orders_clean['order_purchase_timestamp'].dt.year
orders_clean['purchase_month'] = orders_clean['order_purchase_timestamp'].dt.month
orders_clean['purchase_quarter'] = orders_clean['order_purchase_timestamp'].dt.quarter
orders_clean['purchase_dow']   = orders_clean['order_purchase_timestamp'].dt.day_name()

# --- 7. Verify ---
print("\nShape after cleaning:", orders_clean.shape)
print("\nSLA status distribution:")
print(orders_clean['sla_status'].value_counts())
print("\nDelay stats (days):")
print(orders_clean['delay_days'].describe())
print("\nPipeline stage averages (days):")
print(f"  Approval:   {orders_clean['approval_days'].mean():.2f}")
print(f"  Processing: {orders_clean['seller_processing_days'].mean():.2f}")
print(f"  Transit:    {orders_clean['transit_days'].mean():.2f}")
print(f"  Total:      {orders_clean['days_to_deliver'].mean():.2f}")

Timestamps converted successfully
Delivered orders: 96,478
After dropping missing delivery dates: 96,469

Shape after cleaning: (96469, 18)

SLA status distribution:
sla_status
On Time     88644
Breached     5163
At Risk      2662
Name: count, dtype: int64

Delay stats (days):
count    96469.000000
mean       -11.876074
std         10.181995
min       -147.000000
25%        -17.000000
50%        -12.000000
75%         -7.000000
max        188.000000
Name: delay_days, dtype: float64

Pipeline stage averages (days):
  Approval:   0.26
  Processing: 2.74
  Transit:    8.88
  Total:      12.09


/tmp/ipykernel_678/2441633446.py:33: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  orders_clean['days_to_deliver'] = (
/tmp/ipykernel_678/2441633446.py:39: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  orders_clean['delay_days'] = (
/tmp/ipykernel_678/2441633446.py:45: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide

In [5]:
print("Shape:", orders_clean.shape)

print("\nSLA status distribution:")
print(orders_clean['sla_status'].value_counts())

print("\nDelay stats:")
print(orders_clean['delay_days'].describe().round(2))

print("\nPipeline stage averages (days):")
print(f"  Approval:   {orders_clean['approval_days'].mean():.2f}")
print(f"  Processing: {orders_clean['seller_processing_days'].mean():.2f}")
print(f"  Transit:    {orders_clean['transit_days'].mean():.2f}")
print(f"  Total:      {orders_clean['days_to_deliver'].mean():.2f}")


Shape: (96469, 18)

SLA status distribution:
sla_status
On Time     88644
Breached     5163
At Risk      2662
Name: count, dtype: int64

Delay stats:
count    96469.00
mean       -11.88
std         10.18
min       -147.00
25%        -17.00
50%        -12.00
75%         -7.00
max        188.00
Name: delay_days, dtype: float64

Pipeline stage averages (days):
  Approval:   0.26
  Processing: 2.74
  Transit:    8.88
  Total:      12.09


In [6]:
# --- 1. Translate product categories to English ---
category_translation = {
    'cama_mesa_banho': 'bed_bath_table',
    'esporte_lazer': 'sports_leisure',
    'moveis_decoracao': 'furniture_decor',
    'beleza_saude': 'beauty_health',
    'utilidades_domesticas': 'housewares',
    'automotivo': 'automotive',
    'informatica_acessorios': 'computers_accessories',
    'brinquedos': 'toys',
    'relogios_presentes': 'watches_gifts',
    'telefonia': 'telephony',
    'ferramentas_jardim': 'garden_tools',
    'cool_stuff': 'cool_stuff',
    'perfumaria': 'perfumery',
    'bebes': 'baby',
    'eletrodomesticos': 'appliances',
    'eletronicos': 'electronics',
    'fashion_bolsas_e_acessorios': 'fashion_bags_accessories',
    'livros_tecnicos': 'technical_books',
    'construcao_ferramentas_seguranca': 'construction_safety_tools',
    'malas_acessorios': 'luggage_accessories',
    'musica': 'music',
    'papelaria': 'stationery',
    'alimentos_bebidas': 'food_beverages',
    'tablets_impressao_imagem': 'tablets_printing_image',
    'climatizacao': 'air_conditioning',
    'livros_interesse_geral': 'general_books',
    'construcao_ferramentas_construcao': 'construction_tools',
    'pc_gamer': 'gaming_pc',
    'fashion_roupa_masculina': 'mens_fashion',
    'eletrodomesticos_2': 'appliances_2',
    'agro_industria_e_comercio': 'agro_industry',
    'sinalizacao_e_seguranca': 'signaling_security',
    'fashion_underwear_e_moda_praia': 'underwear_beachwear',
    'fashion_esporte': 'sports_fashion',
    'artes': 'arts',
    'market_place': 'marketplace',
    'industria_comercio_e_negocios': 'industry_commerce',
    'dvds_blu_ray': 'dvds_blu_ray',
    'portateis_cozinha_e_preparadores': 'portable_kitchen',
    'cine_foto': 'cinema_photo',
    'consoles_games': 'consoles_games',
    'fraldas_higiene': 'diapers_hygiene',
    'fashion_roupa_feminina': 'womens_fashion',
    'alimentos': 'food',
    'artes_e_artesanato': 'arts_crafts',
    'audio': 'audio',
    'casa_conforto': 'home_comfort',
    'casa_construcao': 'home_construction',
    'casa_conforto_2': 'home_comfort_2',
    'livros_importados': 'imported_books',
    'la_cuisine': 'cuisine',
    'flores': 'flowers',
    'seguros_e_servicos': 'insurance_services',
    'portateis_casa_forno_e_cafe': 'portable_home_oven_coffee'
}

products['category_english'] = products['product_category_name'].map(
    category_translation).fillna(products['product_category_name'])

# --- 2. Merge all tables into one master table ---
master = orders_clean.merge(
    items[['order_id','seller_id','product_id','price','freight_value']],
    on='order_id', how='inner'
).merge(
    sellers[['seller_id','seller_city','seller_state']],
    on='seller_id', how='left'
).merge(
    products[['product_id','category_english','product_weight_g']],
    on='product_id', how='left'
).merge(
    reviews[['order_id','review_score']],
    on='order_id', how='left'
).merge(
    customers[['customer_id','customer_city','customer_state']],
    on='customer_id', how='left'
)

print("Master table shape:", master.shape)
print("\nColumns:", master.columns.tolist())
print("\nNull counts in key columns:")
print(master[['seller_id','category_english',
              'review_score','delay_days','sla_status']].isnull().sum())
print("\nSample:")
print(master[['order_id','seller_id','category_english',
              'delay_days','sla_status','review_score',
              'seller_state','customer_state']].head(3))

Master table shape: (110831, 29)

Columns: ['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'days_to_deliver', 'delay_days', 'seller_processing_days', 'transit_days', 'approval_days', 'sla_status', 'purchase_year', 'purchase_month', 'purchase_quarter', 'purchase_dow', 'seller_id', 'product_id', 'price', 'freight_value', 'seller_city', 'seller_state', 'category_english', 'product_weight_g', 'review_score', 'customer_city', 'customer_state']

Null counts in key columns:
seller_id              0
category_english    1545
review_score         827
delay_days             0
sla_status             0
dtype: int64

Sample:
                           order_id                         seller_id  \
0  e481f51cbdc54678b7cc49136f2d6af7  3504c0cb71d7fa48d967e0e4c94d59d9   
1  53cdb2fc8bc7dce0b6741e2150273451  289cdb325fb7e7f891c38608bf9e0962   
2  47770eb9100c2d0c

In [7]:
# --- 1. Fill remaining nulls ---
master['category_english'] = master['category_english'].fillna('uncategorized')
master['product_weight_g'] = master['product_weight_g'].fillna(
    master['product_weight_g'].median())

# --- 2. Convert sla_status from category to string for MySQL ---
master['sla_status'] = master['sla_status'].astype(str)

# --- 3. Select final columns for MySQL ---
final_cols = [
    'order_id', 'customer_id', 'seller_id',
    'order_purchase_timestamp', 'order_approved_at',
    'order_delivered_carrier_date', 'order_delivered_customer_date',
    'order_estimated_delivery_date',
    'days_to_deliver', 'delay_days', 'seller_processing_days',
    'transit_days', 'approval_days', 'sla_status',
    'purchase_year', 'purchase_month', 'purchase_quarter', 'purchase_dow',
    'price', 'freight_value',
    'category_english', 'product_weight_g',
    'seller_city', 'seller_state',
    'customer_city', 'customer_state',
    'review_score'
]

df_final = master[final_cols].copy()

# --- 4. Export to CSV ---
df_final.to_csv('supply_chain_master.csv', index=False)
print("Exported: supply_chain_master.csv")
print("Shape:", df_final.shape)
print("\nFinal null counts:")
print(df_final.isnull().sum()[df_final.isnull().sum() > 0])
print("\nSLA distribution in final table:")
print(df_final['sla_status'].value_counts())
print("\nColumn list:")
print(df_final.columns.tolist())

Exported: supply_chain_master.csv
Shape: (110831, 27)

Final null counts:
order_approved_at     15
approval_days         15
review_score         827
dtype: int64

SLA distribution in final table:
sla_status
On Time     102077
Breached      5746
At Risk       3008
Name: count, dtype: int64

Column list:
['order_id', 'customer_id', 'seller_id', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'days_to_deliver', 'delay_days', 'seller_processing_days', 'transit_days', 'approval_days', 'sla_status', 'purchase_year', 'purchase_month', 'purchase_quarter', 'purchase_dow', 'price', 'freight_value', 'category_english', 'product_weight_g', 'seller_city', 'seller_state', 'customer_city', 'customer_state', 'review_score']
